# 18 — File Storage (buckets, `FileRef`, `FileField`)

> **experimental: follows SurrealDB's experimental `files` capability.** A change of syntax when SurrealDB
> stabilises file storage will ship as a patch release of the ORM.

SurrealDB stores files in **buckets** and records point at them with a **file pointer** (`f"bucket:/key"` in
SurrealQL). The ORM covers the whole path:

- **SDK** — `FileRef(bucket, key)` (the pointer, CBOR tag 55), `FileInfo`, and nine methods on every connection:
  `file_put()`, `file_get()`, `file_head()`, `file_exists()`, `file_delete()`, `file_copy()`, `file_rename()`,
  `file_list()`, `iter_files()`.
- **ORM** — `FileField["bucket"]` / `ImageField["bucket"]` columns (`option<file<bucket>>`) and the model helpers
  `upload_file()`, `download_file()`, `file_info()`, `delete_file()`.
- **Migrations** — `DefineBucket`, `AlterBucket`, `RemoveBucket`, introspected and diffed like tables.

**Requirements and warnings**

1. The server must be started with **`--allow-experimental=files`** (or `SURREAL_CAPS_ALLOW_EXPERIMENTAL=files`), on
   SurrealDB 3.2.x and 3.3.x. The project's `devops/docker-compose.yml` does it. Without it every file call raises
   `FilesNotEnabledError`.
2. **A bucket defined without `PERMISSIONS` is `FULL`**: every record user can read and write every file in it.
   Section 5 shows a rule on `$action`.
3. Files are **not transactional**: a cancelled transaction does not undo a file write, so the helpers take no `tx=`.
4. A payload is sent whole and held in memory: up to 4 MiB over HTTP and 128 MiB over WebSocket by default (the
   server's limits). `upload_file()` switches to the WebSocket connection by itself above the HTTP limit.
5. File storage needs the CBOR protocol (the default); a JSON connection refuses file calls.

## Setup

One connection, its own database. The WebSocket connection that large uploads use is created from the same settings
when it is first needed.

In [ ]:
import os

from surreal_orm import SurrealDBConnectionManager

HOST = os.environ.get("SURREALDB_HOST", "localhost")
PORT = os.environ.get("SURREALDB_PORT", "8000")

SurrealDBConnectionManager.set_connection(
    url=f"http://{HOST}:{PORT}",
    user="root",
    password="root",
    namespace="examples",
    database="examples_files",
)
print("Connection configured.")
# Connection configured.

---

## 1. Buckets in migrations (offline)

`DefineBucket` writes `DEFINE BUCKET`; `AlterBucket` redefines one (`DEFINE BUCKET OVERWRITE`) and rolls back to the
previous definition it was given (the `previous_*` fields; `previous_permissions` makes it reversible); `RemoveBucket` is irreversible (the files go with the bucket), so `makemigrations`
holds it back unless `--drop-missing` is passed. A permission rule is written **without** `WHERE`.

`parse_define_bucket()` reads the text `INFO FOR DB` reports, which is how `DatabaseIntrospector` compares the
database with your migrations.

In [ ]:
from surreal_orm.migrations.define_parser import parse_define_bucket
from surreal_orm.migrations.operations import AlterBucket, DefineBucket, RemoveBucket

define = DefineBucket("avatars", backend="memory", permissions="$action INSIDE ['get', 'head']", comment="it's")
print(define.forwards())
print(define.backwards())
# DEFINE BUCKET avatars BACKEND 'memory' PERMISSIONS WHERE $action INSIDE ['get', 'head'] COMMENT 'it\'s';
# REMOVE BUCKET avatars;

alter = AlterBucket("avatars", backend="memory", readonly=True, previous_backend="memory", previous_permissions="FULL")
print(alter.forwards())
print(alter.backwards())
# DEFINE BUCKET OVERWRITE avatars READONLY BACKEND 'memory';
# DEFINE BUCKET OVERWRITE avatars BACKEND 'memory' PERMISSIONS FULL;

remove = RemoveBucket("old-uploads")
print(remove.forwards(), remove.reversible)
# REMOVE BUCKET `old-uploads`; False

print(parse_define_bucket("DEFINE BUCKET avatars BACKEND 'memory' PERMISSIONS FULL"))
# BucketState(name='avatars', backend='memory', readonly=False, permissions='FULL', comment=None)

---

## 2. `FileRef` — the pointer (offline)

A `FileRef` is a bucket name and a key. The key always starts with `/` (added when missing), and a key with a `.` or
`..` segment, a NUL or nothing below the root is refused, whatever the backend. `FileRef.parse()` reads the
`"bucket:/key"` text (with or without the `f"..."` wrapper), and `str()` gives it back.

In [ ]:
from surreal_orm import FileRef

ref = FileRef("docs", "reports/2026/q1.pdf")
print(repr(ref))
print(str(ref))
print(FileRef.parse('f"docs:/reports/2026/q1.pdf"') == ref)
print(ref.with_key("reports/2026/q2.pdf"))
# FileRef(bucket='docs', key='/reports/2026/q1.pdf')
# docs:/reports/2026/q1.pdf
# True
# docs:/reports/2026/q2.pdf

try:
    FileRef("docs", "reports/../secrets.txt")
except ValueError as exc:
    print(exc)
# Invalid file key 'reports/../secrets.txt': it must name a file below the bucket root (no empty path, no empty, '.' or '..' segment, no NUL)

---

## 3. File fields on a model (offline)

`FileField["bucket"]` holds a `FileRef` restricted to the buckets named (the first one is where uploads go by
default); `ImageField["bucket"]` also checks that uploaded data starts with a PNG, JPEG, GIF or WebP signature. Both
are optional with a default of `None`, accept the `"bucket:/key"` text, and refuse a pointer to another bucket.
Migrations map them to `option<file<bucket>>` (type `file<bucket>`, nullable).

In [ ]:
from pydantic import ValidationError

from surreal_orm import BaseSurrealModel, FileField, ImageField, SurrealConfigDict
from surreal_orm.migrations.introspector import ModelIntrospector


class NBFileDoc(BaseSurrealModel):
    model_config = SurrealConfigDict(table_name="nb_files_doc")

    id: str | None = None
    title: str = ""
    attachment: FileField["nb_files_docs"]  # noqa: F821
    cover: ImageField["nb_files_images"]  # noqa: F821


doc = NBFileDoc(title="draft", attachment="nb_files_docs:/drafts/a.txt")
print(repr(doc.attachment), doc.cover)
# FileRef(bucket='nb_files_docs', key='/drafts/a.txt') None

try:
    NBFileDoc(attachment=FileRef("nb_files_images", "/a.png"))
except ValidationError as exc:
    print(exc.errors()[0]["msg"])
# File pointer nb_files_images:/a.png is not in the allowed bucket(s) nb_files_docs

fields = ModelIntrospector([NBFileDoc]).introspect().tables["nb_files_doc"].fields
for name in ("attachment", "cover"):
    print(name, fields[name].field_type, "nullable" if fields[name].nullable else "required")
# attachment file<nb_files_docs> nullable
# cover file<nb_files_images> nullable

---

## 4. Buckets and files through the SDK (live)

The buckets here use `BACKEND 'memory'`: their files live in the server's memory. A `file:/dir` backend also needs
the directory in the server's `SURREAL_BUCKET_FOLDER_ALLOWLIST` (otherwise `File access denied`); neither the compose
file nor this notebook sets one. The cell starts by removing what a previous run left.

In [ ]:
import contextlib

client = await SurrealDBConnectionManager.get_client()

BUCKETS = ("nb_files_docs", "nb_files_images", "nb_files_private")
await client.query("REMOVE TABLE IF EXISTS nb_files_doc;")
for name in BUCKETS:
    with contextlib.suppress(Exception):
        await client.query(f"REMOVE BUCKET IF EXISTS {name};")

for op in (DefineBucket("nb_files_docs", backend="memory"), DefineBucket("nb_files_images", backend="memory")):
    await client.query(op.forwards())

notes = FileRef("nb_files_docs", "/notes/hello.txt")
await client.file_put(notes, b"Hello, buckets!")
print(await client.file_get(notes))
print(await client.file_exists(notes), (await client.file_head(notes)).size)
# b'Hello, buckets!'
# True 15

copy = await client.file_copy(notes, "/notes/copy.txt")
print([info.key for info in await client.file_list("nb_files_docs", prefix="/notes/")])
await client.file_delete(copy)
print(await client.file_get(copy))
# ['/notes/copy.txt', '/notes/hello.txt']
# None

---

## 5. Model helpers (live)

`upload_file(field, data)` writes the file under `/<table>/<id>/<column>` (or `key=`; the id is percent-encoded, so
`a/b` gives `a%2Fb`), in the field's first bucket
(or `bucket=`), then sets the column through the same path as `merge()`: update signals, cache invalidation,
`QueryLogger`. The record must be saved first. `download_file()` reads the content, `file_info()` the size and last
write, and `delete_file()` deletes the file, then removes the column.

If the record update fails after the file was written, the error is raised and the file stays.

In [ ]:
doc = NBFileDoc(id="report", title="Quarterly report")
await doc.save()

ref = await doc.upload_file("attachment", b"Q1 revenue: 42")
print(ref)
print(await doc.download_file("attachment"))
info = await doc.file_info("attachment")
print(info.size, type(info.updated).__name__)
# nb_files_docs:/nb_files_doc/report/attachment
# b'Q1 revenue: 42'
# 14 datetime

reloaded = await NBFileDoc.objects().get("report")
print(reloaded.attachment == ref)
print([d.id for d in await NBFileDoc.objects().filter(attachment=str(ref)).exec()])
# True
# ['report']

try:
    await doc.upload_file("cover", b"not an image")
except ValueError as exc:
    print(exc)
# NBFileDoc.cover accepts PNG, JPEG, GIF or WebP data only

png = b"\x89PNG\r\n\x1a\n" + b"\x00" * 16
print(await doc.upload_file("cover", png))
# nb_files_images:/nb_files_doc/report/cover

await doc.delete_file("attachment")
print(doc.attachment, await client.file_exists(ref))
# None False

---

## 6. Permissions, cleanup on delete, `overwrite=False` (live)

**Permissions.** A bucket takes one rule for every action: an expression on `$action` (`put`, `get`, `head`,
`delete`, `copy`, `rename`, `exists`, `list`), `$file` and `$auth`, written without `WHERE`. This one lets record
users read and nothing else (root users are not subject to it).

**Cleanup.** Deleting a record does not delete its files. A `DefineEvent` on the table does it in the database.

**`overwrite=False`.** `upload_file(..., overwrite=False)` checks the key first and raises `FileExistsError` when a
file is there; nothing is written and the record is unchanged. A writer that creates the file between the check and
the upload still wins (its content is kept and no error is raised): use a unique `key=` when that matters.

In [ ]:
from surreal_orm.migrations.operations import DefineEvent

private = DefineBucket("nb_files_private", backend="memory", permissions="$action INSIDE ['get', 'head', 'exists', 'list']")
await client.query(private.forwards())
info_db = (await client.query("INFO FOR DB;")).first_result.result
print(parse_define_bucket(info_db["buckets"]["nb_files_private"]).permissions)
# $action INSIDE ['get', 'head', 'exists', 'list']

cleanup = DefineEvent(
    name="nb_files_doc_cleanup",
    table="nb_files_doc",
    when="$event = 'DELETE' AND $before.cover != NONE",
    then="$before.cover.delete()",
)
await client.query(cleanup.forwards())
cover = doc.cover
await doc.delete()
print(await client.file_exists(cover))
# False

draft = NBFileDoc(id="draft", title="Draft")
await draft.save()
await draft.upload_file("attachment", b"first version", overwrite=False)
try:
    await draft.upload_file("attachment", b"second version", overwrite=False)
except FileExistsError as exc:
    print(exc)
print(await draft.download_file("attachment"))
# nb_files_docs:/nb_files_doc/draft/attachment already exists: pass overwrite=True or another key=
# b'first version'

---

## Cleanup

Removing a bucket removes its files (`RemoveBucket(...).forwards()`).

In [ ]:
await client.query("REMOVE TABLE IF EXISTS nb_files_doc;")
for name in BUCKETS:
    await client.query(RemoveBucket(name).forwards())
print(sorted((await client.query("INFO FOR DB;")).first_result.result["buckets"]))
await SurrealDBConnectionManager.close_connection()
# []